Importation des données en forme de DataFrame pandas

In [ ]:
import pandas as pd
import numpy as np
import os

# Liste vide pour stocker les lignes
data = []

# Exemple de boucle sur les fichiers
for subject_id in range(1, 11):  # 10 sujets
    for digit in range(10):      # 0 à 9
        for repetition in range(1, 11):  # 10 répétitions de 1 à 10
            # Ex: "subject1_digit2_rep5.txt"
            filename = f"Subject{subject_id}-{digit}-{repetition}.csv"
            filepath = os.path.join("Domain1_csv", filename)

            if os.path.exists(filepath):
                # Charger uniquement x, y, z (on ignore t)
                points = np.loadtxt(filepath, delimiter=",", skiprows=1, usecols=(0,1,2))
                
                data.append({
                    "subject_id": subject_id,
                    "digit": digit,
                    "repetition": repetition,
                    "points": points
                })
            else:
                print("Fichier manquant :", filepath)    

# Convertir en DataFrame
df = pd.DataFrame(data)



In [ ]:
# Vérification des résultats de la base de données
print(df.head())   


gesture_row = df[
    (df["subject_id"] == 6) &
    (df["digit"] == 3) &
    (df["repetition"] == 7)
]

#points = gesture_row.iloc[0]["points"]
#print(points)

Centrage et normalisation des points

In [ ]:
def center_and_normalize(gesture):
    center = np.mean(gesture, axis=0) # Calcule moyenne par colonne x,y,z
    centered = gesture - center
    norm = np.linalg.norm(centered)  # Cacul de la norme des points
    return centered / norm

# Appliquer la fonction center_and_normalize --> Ajoute une colonne points_normalized 
df["points_normalized"] = df["points"].apply(center_and_normalize)

# Vérification des premier points
df[["subject_id", "digit", "repetition", "points_normalized"]].head()


In [ ]:
# Rassemblment des points normaliser 
from sklearn.cluster import KMeans
import numpy as np

# On empile tous les points normalisés dans un grand tableau
all_points = np.vstack(df["points_normalized"].values)

In [ ]:
# Méthode du coude pour trouver le nombre de cluster optimal
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

inertia_values = []
k_values = list(range(2, 31))  # Essaye k de 2 à 30

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(all_points)
    inertia_values.append(kmeans.inertia_)  # inertie = somme des distances au centre

# Tracer la courbe
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia_values, marker='o')
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertia (somme des distances intra-clusters)")
plt.title("Méthode du coude pour déterminer le nombre optimal de clusters")
plt.grid(True)
plt.show()

In [ ]:
from kneed import KneeLocator

# Detéction automatique du coude 
kneedle = KneeLocator(k_values, inertia_values, curve="convex", direction="decreasing")
print("Coude détecté à k =", kneedle.elbow)
kneedle.plot_knee()

In [ ]:
# Application du clustering
k = kneedle.elbow
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(all_points)

# Récupération des centres de clusters
centroids = kmeans.cluster_centers_


In [ ]:
def encode_gesture(gesture_points, centroids):
    # Pour chaque point : on calcule la distance à tous les centres
    # et on prend l'index du plus proche
    return [
        np.argmin(np.linalg.norm(point - centroids, axis=1))
        for point in gesture_points
    ]
    
df["encoded_sequence"] = df["points_normalized"].apply(
    #Indique le numéro le plus proche un point x,y,z
    lambda g: encode_gesture(g, centroids)
)    
    

In [ ]:
#Transforme séquence d'entier en chaine de caractère
df["encoded_string"] = df["encoded_sequence"].apply(
    lambda seq: ''.join(chr(65 + i) for i in seq)  # transforme [0,1,2] → "ABC"
    #Attention max 26 clusters avec cet encodage
)

In [ ]:
df[["subject_id", "digit", "repetition", "encoded_string"]].head()

Baseline Method : Edit distance

In [ ]:
def Edit_Distance(stringA,stringB) :
    #stringA = ligne
    #stringB =colonne
    Mat =np.zeros((len(stringA)+1,len(stringB)+1),dtype=int)
    for i in range(len(stringA)+1):
        Mat[i][0]=i
    for j in range(len(stringB)+1):
        Mat[0][j]=j


    for i in range(len(stringA)):
        for j in range(len(stringB)) :
            diag=0
            if stringA[i]!=stringB[j]:
                diag=1
            Mat[i+1][j+1]=min(Mat[i][j]+diag,Mat[i][j+1]+1,Mat[i+1][j]+1)
    print(Mat)
    return Mat[-1][-1]

Validation et comparaison user independant 

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

def leave_one_user_out(df, k=1, output_csv="user_independent_results.csv"):
    accuracies = []
    results = []  # Pour stocker chaque ligne à écrire dans le CSV

    for test_user in range(1, 11):  # 10 utilisateurs
        train_df = df[df["subject_id"] != test_user]
        test_df = df[df["subject_id"] == test_user]

        y_true = []
        y_pred = []

        for _, test_row in test_df.iterrows():
            test_seq = test_row["encoded_string"]
            test_label = test_row["digit"]

            distances = []
            for _, train_row in train_df.iterrows():
                train_seq = train_row["encoded_string"]
                dist = Edit_Distance(test_seq, train_seq)
                distances.append((dist, train_row["digit"]))

            # k-NN
            distances.sort(key=lambda x: x[0])
            top_k = distances[:k]
            predicted = Counter([d[1] for d in top_k]).most_common(1)[0][0]

            y_true.append(test_label)
            y_pred.append(predicted)

        # Accuracy pour ce user
        accuracy = np.mean(np.array(y_true) == np.array(y_pred))
        accuracies.append(accuracy)
        results.append({"user": test_user, "accuracy": accuracy})

    # Moyenne et écart-type
    avg = np.mean(accuracies)
    std = np.std(accuracies)

    # Ajouter ligne moyenne au CSV
    results.append({"user": "average", "accuracy": avg})
    results.append({"user": "std", "accuracy": std})

    # Sauvegarde dans un CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)

    return accuracies, avg, std

accuracies, average, std_dev = leave_one_user_out(df, k=1)


In [ ]:
#User dependant 
import pandas as pd
import numpy as np
from collections import Counter

def user_dependent_validation(df, k=1, output_csv="user_dependent_results.csv"):
    results = []

    for user in range(1, 11):  # Utilisateur de 1 à 10
        user_df = df[df["subject_id"] == user]
        accuracies_per_fold = []

        for fold in range(10):  # 10-fold cross-validation
            train_rows = []
            test_rows = []

            # Pour chaque chiffre (0 à 9), on prend 1 exemple pour le test
            for digit in range(10):
                digit_df = user_df[user_df["digit"] == digit]
                digit_df = digit_df.sample(frac=1, random_state=fold)  # shuffle
                test_row = digit_df.iloc[fold % 10]  # 1 geste pour test
                test_rows.append(test_row)
                train_rows.extend(digit_df.drop(test_row.name).values.tolist())  # les 9 restants pour entraînement

            # Conversion en DataFrames
            test_df = pd.DataFrame(test_rows, columns=user_df.columns)
            train_df = pd.DataFrame(train_rows, columns=user_df.columns)

            y_true, y_pred = [], []

            for _, test_row in test_df.iterrows():
                test_seq = test_row["encoded_string"]
                test_label = test_row["digit"]

                distances = []
                for _, train_row in train_df.iterrows():
                    train_seq = train_row["encoded_string"]
                    dist = Edit_Distance(test_seq, train_seq)
                    distances.append((dist, train_row["digit"]))

                distances.sort(key=lambda x: x[0])
                top_k = distances[:k]
                predicted = Counter([d[1] for d in top_k]).most_common(1)[0][0]

                y_true.append(test_label)
                y_pred.append(predicted)

            accuracy = np.mean(np.array(y_true) == np.array(y_pred))
            accuracies_per_fold.append(accuracy)

        mean_acc = np.mean(accuracies_per_fold)
        std_acc = np.std(accuracies_per_fold)
        results.append({"user": user, "mean_accuracy": mean_acc, "std_accuracy": std_acc})

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)

    return results_df

results = user_dependent_validation(df, k=1)
